# Multi-agent Collaboration for Financial Analysis

In this lesson, you will learn ways for making agents collaborate with each other.

کتابخانه‌ها در محیط کلاس از قبل نصب شده‌اند. اگر این نوت‌بوک را روی سیستم خودتان اجرا می‌کنید:
```Python
!pip install crewai crewai-tools python-dotenv
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import libraries, APIs and LLM

In [4]:
from crewai import Agent, Task, Crew, Process, LLM

**Note**: 
- The video uses `gpt-4-turbo`, but due to certain constraints, and in order to offer this course for free to everyone, the code you'll run here will use `gpt-3.5-turbo`.
- You can use `gpt-4-turbo` when you run the notebook _locally_ (using `gpt-4-turbo` will not work on the platform)
- Thank you for your understanding!

In [14]:
import os
from dotenv import load_dotenv

load_dotenv()

# تنظیم مدل زبانی
llm = LLM(
    model="gpt-4o-mini",
)


## crewAI Tools

In [16]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool# TavilySearchTool

search_tool = SerperDevTool()# TavilySearchTool()
scrape_tool = ScrapeWebsiteTool()

## Creating Agents

In [18]:
data_analyst_agent = Agent(
    role="تحلیلگر داده بازار سرمایه",
    goal="پایش و تحلیل لحظه‌ای داده‌های بازار بورس تهران "
         "برای شناسایی روندها و پیش‌بینی تحرکات قیمتی.",
    backstory=(
        "متخصص در بازارهای مالی ایران، از مدل‌سازی آماری "
        "و یادگیری ماشین برای استخراج بینش‌های کلیدی استفاده می‌کند. "
        "ستون فقرات تصمیم‌گیری معاملاتی در تیم است."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=5,
    tools=[scrape_tool, search_tool]
)

In [22]:
trading_strategy_agent = Agent(
    role="توسعه‌دهنده استراتژی معاملاتی",
    goal="طراحی و آزمون استراتژی‌های معاملاتی "
         "بر اساس تحلیل‌های تحلیلگر داده.",
    backstory=(
        "با درک عمیق از بازار بورس تهران و تحلیل کمّی، "
        "استراتژی‌های معاملاتی را طراحی و بهینه می‌کند. "
        "رویکردهای مختلف را ارزیابی کرده و سودآورترین "
        "و کم‌ریسک‌ترین گزینه را انتخاب می‌کند."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=5,
    tools=[scrape_tool, search_tool]
)

In [24]:
execution_agent = Agent(
    role="مشاور اجرای معامله",
    goal="پیشنهاد بهترین روش اجرای معاملات "
         "بر اساس استراتژی‌های تأییدشده.",
    backstory=(
        "متخصص در تحلیل زمان‌بندی، قیمت و جزئیات لجستیکی معاملات. "
        "با ارزیابی این عوامل، پیشنهادهای مستدلی برای "
        "زمان و نحوه اجرای معامله ارائه می‌دهد."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=5,
    tools=[scrape_tool, search_tool]
)

In [26]:
risk_management_agent = Agent(
    role="مشاور مدیریت ریسک",
    goal="ارزیابی و ارائه بینش درباره ریسک‌های "
         "مرتبط با فعالیت‌های معاملاتی پیشنهادی.",
    backstory=(
        "مجهز به درک عمیق از مدل‌های ارزیابی ریسک و دینامیک بازار، "
        "ریسک‌های بالقوه معاملات پیشنهادی را بررسی می‌کند. "
        "تحلیل دقیق ریسک ارائه داده و پیشنهاداتی برای "
        "حفظ انطباق با سطح تحمل ریسک شرکت می‌دهد."
    ),
    verbose=True,
    allow_delegation=True,
    llm=llm,
    max_iter=5,
    tools=[scrape_tool, search_tool]
)

## Creating Tasks

In [29]:
# تسک تحلیلگر داده: تحلیل داده‌های بازار
data_analysis_task = Task(
    description=(
        "داده‌های بازار بورس تهران را برای نماد ({stock_selection}) "
        "پایش و تحلیل کن. "
        "از مدل‌سازی آماری برای شناسایی روندها "
        "و پیش‌بینی تحرکات قیمتی استفاده کن."
    ),
    expected_output=(
        "بینش‌ها و هشدارهای مهم درباره فرصت‌ها "
        "یا تهدیدات بازار برای نماد {stock_selection}."
    ),
    agent=data_analyst_agent,
)

In [31]:
# تسک توسعه استراتژی معاملاتی
strategy_development_task = Task(
    description=(
        "بر اساس تحلیل‌های تحلیلگر داده و "
        "سطح تحمل ریسک تعریف‌شده ({risk_tolerance})، "
        "استراتژی‌های معاملاتی را توسعه و اصلاح کن. "
        "رویکرد معاملاتی مورد نظر را هم در نظر بگیر ({trading_strategy_preference})."
    ),
    expected_output=(
        "مجموعه‌ای از استراتژی‌های معاملاتی بالقوه برای نماد {stock_selection} "
        "که با سطح تحمل ریسک کاربر همخوانی دارند."
    ),
    agent=trading_strategy_agent,
)

In [33]:
# تسک برنامه‌ریزی اجرای معامله
execution_planning_task = Task(
    description=(
        "استراتژی‌های معاملاتی تأییدشده را برای نماد {stock_selection} "
        "تحلیل کن و بهترین روش‌های اجرا را "
        "با توجه به شرایط فعلی بازار و قیمت‌گذاری بهینه مشخص کن."
    ),
    expected_output=(
        "برنامه‌های اجرایی دقیق با پیشنهاد زمان و نحوه "
        "انجام معاملات برای نماد {stock_selection}."
    ),
    agent=execution_agent,
)

In [35]:
# تسک ارزیابی ریسک معاملات
risk_assessment_task = Task(
    description=(
        "ریسک‌های مرتبط با استراتژی‌های معاملاتی "
        "و برنامه‌های اجرایی پیشنهادشده برای نماد {stock_selection} را ارزیابی کن. "
        "تحلیل دقیقی از ریسک‌های احتمالی ارائه بده "
        "و استراتژی‌های کاهش ریسک را پیشنهاد کن."
    ),
    expected_output=(
        "گزارش جامع تحلیل ریسک با جزئیات ریسک‌های احتمالی "
        "و توصیه‌های کاهش ریسک برای نماد {stock_selection}."
    ),
    agent=risk_management_agent,
)

## Creating the Crew
- The `Process` class helps to delegate the workflow to the Agents (kind of like a Manager at work)
- In the example below, it will run this hierarchically.
- `manager_llm` lets you choose the "manager" LLM you want to use.

In [38]:
# تعریف Crew با فرایند سلسله‌مراتبی
financial_trading_crew = Crew(
    agents=[data_analyst_agent,
            trading_strategy_agent,
            execution_agent,
            risk_management_agent],

    tasks=[data_analysis_task,
           strategy_development_task,
           execution_planning_task,
           risk_assessment_task],

    manager_llm=llm,
    process=Process.hierarchical,
    verbose=True
)

## Running the Crew

- Set the inputs for the execution of the crew.

In [40]:
# داده‌های ورودی برای اجرای crew
financial_trading_inputs = {
    "stock_selection": "فولاد",       # نماد فولاد مبارکه اصفهان
    "initial_capital": "500000000",   # ۵۰۰ میلیون تومان
    "risk_tolerance": "متوسط",
    "trading_strategy_preference": "نوسان‌گیری کوتاه‌مدت",
    "news_impact_consideration": True
}

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [43]:
### this execution will take some time to run
result = financial_trading_crew.kickoff(inputs=financial_trading_inputs)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.2                                                                                        │
│  Latest version:  1.14.5                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: c1664d4c-3060-4f58-a5cf-3ed614a21829                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: داده‌های بازار بورس تهران را برای نماد (فولاد) پایش و تحلیل کن. از مدل‌سازی آماری برای شناسایی روندها و    │
│  پیش‌بینی تحرکات قیمتی استفاده کن.                                                                               │
│  ID: b1a7bc9f-b02d-4c01-9cf4-d1018393fcb5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: داده‌های بازار بورس تهران را برای نماد (فولاد) پایش و تحلیل کن. از مدل‌سازی آماری برای شناسایی روندها و    │
│  پیش‌بینی تحرکات قیمتی استفاده کن.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ERROR:root:Failed to connect to OpenAI API: Request timed out.
ERROR:root:OpenAI API call failed: Failed to connect to OpenAI API: Request timed out.


...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ### تحلیل ریسک‌های مرتبط با استراتژی‌های معاملاتی نماد فولاد                                                     │
│                                                                                                                 │
│  #### 1. ریسک‌های نوسانات قیمت:                                                                                  │
│  - **توصیف**: قیمت فولاد تحت تأثیر عوامل بازار، عرضه و تقاضا، و شرایط اقتصادی جهانی قرار دارد و ممکن است        │
│  نوسانات شدیدی را تجربه کند.                                                                                    │
│  - **استراتژی کاهش ریسک**: استفاده از استراتژی‌های هجینگ مانند قراردادهای آتی یا گزینه‌ها برای محافظت در برابر    │
│  نوسانات قیمتی و تضمین حداقل سود.                                                                               │
│                                                                                                                 │
│  #### 2. عوامل کلان اقتصادی:                                                                                    │
│  - **توصیف**: وضعیت اقتصادی کلی، شامل تورم، نرخ بهره و شرایط تجاری، می‌تواند بر قیمت فولاد و بازار سرمایه تأثیر  │
│  بگذارد.                                                                                                        │
│  - **استراتژی کاهش ریسک**: تجزیه و تحلیل دقیق و به‌روز متغیرهای کلان اقتصادی و تنظیم استراتژی‌های معاملاتی بر     │
│  اساس این عوامل برای جلوگیری از آسیب‌پذیری.                                                                      │
│                                                                                                                 │
│  #### 3. ریسک‌های مربوط به تأمین تجهیزات و مواد اولیه:                                                           │
│  - **توصیف**: نوسانات در تأمین و قیمت مواد اولیه (مانند سنگ آهن) و تجهیزات تولید می‌تواند تأثیر مستقیم بر        │
│  هزینه‌ها و سودآوری داشته باشد.                                                                                  │
│  - **استراتژی کاهش ریسک**: ایجاد قراردادهای بلندمدت با تأمین‌کنندگان و تنوع بخشی به منابع تأمین برای کاهش        │
│  وابستگی و ریسک‌های ناشی از اختلالات تأمین.                                                                      │
│                                                                                                                 │
│  #### 4. ریسک‌های سیستم و تجهیزات معاملاتی:                                                                      │
│  - **توصیف**: مشکلات در سیستم‌های معاملاتی می‌تواند منجر به خطاهای انسانی یا فوت‌های مالی شود.                     │
│  - **استراتژی کاهش ریسک**: به‌روزرسانی منظم تجهیزات و نرم‌افزارهای معاملاتی، آموزش مستمر کارکنان و ایجاد          │
│  سیستم‌های پشتیبان برای مدیریت بحران.                                                                            │
│                                                                                                                 │
│  #### 5. ریسک‌های مربوط به تغییرات قانونی و مقرراتی:                                                             │
│  - **توصیف**: تغییرات در قوانین و مقررات اقتصادی می‌تواند بر عملکرد شرکت تأثیرگذار باشد.                         │
│  - **استراتژی کاهش ریسک**: رصد مداوم تغییرات قانونی و مشاوره حقوقی برای انطباق با الزامات جدید و جلوگیری از     │
│  جریمه‌های مالی.                                                                                                 │
│                                

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: ریسک‌های مرتبط با استراتژی‌های معاملاتی و برنامه‌های اجرایی پیشنهادشده برای نماد فولاد را ارزیابی کن.       │
│  تحلیل دقیقی از ریسک‌های احتمالی ارائه بده و استراتژی‌های کاهش ریسک را پیشنهاد کن.                                │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_completed' emitted with empty scope stack. Missing starting 
event?

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: c1664d4c-3060-4f58-a5cf-3ed614a21829                                                                       │
│  Final Output: ### تحلیل ریسک‌های مرتبط با استراتژی‌های معاملاتی نماد فولاد                                       │
│                                                                                                                 │
│  #### 1. ریسک‌های نوسانات قیمت:                                                                                  │
│  - **توصیف**: قیمت فولاد تحت تأثیر عوامل بازار، عرضه و تقاضا، و شرایط اقتصادی جهانی قرار دارد و ممکن است        │
│  نوسانات شدیدی را تجربه کند.                                                                                    │
│  - **استراتژی کاهش ریسک**: استفاده از استراتژی‌های هجینگ مانند قراردادهای آتی یا گزینه‌ها برای محافظت در برابر    │
│  نوسانات قیمتی و تضمین حداقل سود.                                                                               │
│                                                                                                                 │
│  #### 2. عوامل کلان اقتصادی:                                                                                    │
│  - **توصیف**: وضعیت اقتصادی کلی، شامل تورم، نرخ بهره و شرایط تجاری، می‌تواند بر قیمت فولاد و بازار سرمایه تأثیر  │
│  بگذارد.                                                                                                        │
│  - **استراتژی کاهش ریسک**: تجزیه و تحلیل دقیق و به‌روز متغیرهای کلان اقتصادی و تنظیم استراتژی‌های معاملاتی بر     │
│  اساس این عوامل برای جلوگیری از آسیب‌پذیری.                                                                      │
│                                                                                                                 │
│  #### 3. ریسک‌های مربوط به تأمین تجهیزات و مواد اولیه:                                                           │
│  - **توصیف**: نوسانات در تأمین و قیمت مواد اولیه (مانند سنگ آهن) و تجهیزات تولید می‌تواند تأثیر مستقیم بر        │
│  هزینه‌ها و سودآوری داشته باشد.                                                                                  │
│  - **استراتژی کاهش ریسک**: ایجاد قراردادهای بلندمدت با تأمین‌کنندگان و تنوع بخشی به منابع تأمین برای کاهش        │
│  وابستگی و ریسک‌های ناشی از اختلالات تأمین.                                                                      │
│                                                                                                                 │
│  #### 4. ریسک‌های سیستم و تجهیزات معاملاتی:                                                                      │
│  - **توصیف**: مشکلات در سیستم‌های معاملاتی می‌تواند منجر به خطاهای انسانی یا فوت‌های مالی شود.                     │
│  - **استراتژی کاهش ریسک**: به‌روزرسانی منظم تجهیزات و نرم‌افزارهای معاملاتی، آموزش مستمر کارکنان و ایجاد          │
│  سیستم‌های پشتیبان برای مدیریت بحران.                                                                            │
│                                                                                                                 │
│  #### 5. ریسک‌های مربوط به تغییرات قانونی و مقرراتی:                                                             │
│  - **توصیف**: تغییرات در قوانین و مقررات اقتصادی می‌تواند بر عملکرد شرکت تأثیرگذار باشد.                         │
│  - **استراتژی کاهش ریسک**: رصد مداوم تغییرات قانونی و مشاوره حقوقی برای انطباق با الزامات جدید و جلوگیری از     │
│  جریمه‌های مالی.                                                                                                 │
│                               

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

- Display the final result as Markdown.

In [44]:
from IPython.display import Markdown

content = result.raw if hasattr(result, "raw") else str(result)
content = content.strip().removeprefix("```markdown").removesuffix("```").strip()
Markdown(content)

### تحلیل ریسک‌های مرتبط با استراتژی‌های معاملاتی نماد فولاد

#### 1. ریسک‌های نوسانات قیمت:
- **توصیف**: قیمت فولاد تحت تأثیر عوامل بازار، عرضه و تقاضا، و شرایط اقتصادی جهانی قرار دارد و ممکن است نوسانات شدیدی را تجربه کند.
- **استراتژی کاهش ریسک**: استفاده از استراتژی‌های هجینگ مانند قراردادهای آتی یا گزینه‌ها برای محافظت در برابر نوسانات قیمتی و تضمین حداقل سود.

#### 2. عوامل کلان اقتصادی:
- **توصیف**: وضعیت اقتصادی کلی، شامل تورم، نرخ بهره و شرایط تجاری، می‌تواند بر قیمت فولاد و بازار سرمایه تأثیر بگذارد.
- **استراتژی کاهش ریسک**: تجزیه و تحلیل دقیق و به‌روز متغیرهای کلان اقتصادی و تنظیم استراتژی‌های معاملاتی بر اساس این عوامل برای جلوگیری از آسیب‌پذیری.

#### 3. ریسک‌های مربوط به تأمین تجهیزات و مواد اولیه:
- **توصیف**: نوسانات در تأمین و قیمت مواد اولیه (مانند سنگ آهن) و تجهیزات تولید می‌تواند تأثیر مستقیم بر هزینه‌ها و سودآوری داشته باشد.
- **استراتژی کاهش ریسک**: ایجاد قراردادهای بلندمدت با تأمین‌کنندگان و تنوع بخشی به منابع تأمین برای کاهش وابستگی و ریسک‌های ناشی از اختلالات تأمین.

#### 4. ریسک‌های سیستم و تجهیزات معاملاتی:
- **توصیف**: مشکلات در سیستم‌های معاملاتی می‌تواند منجر به خطاهای انسانی یا فوت‌های مالی شود.
- **استراتژی کاهش ریسک**: به‌روزرسانی منظم تجهیزات و نرم‌افزارهای معاملاتی، آموزش مستمر کارکنان و ایجاد سیستم‌های پشتیبان برای مدیریت بحران.

#### 5. ریسک‌های مربوط به تغییرات قانونی و مقرراتی:
- **توصیف**: تغییرات در قوانین و مقررات اقتصادی می‌تواند بر عملکرد شرکت تأثیرگذار باشد.
- **استراتژی کاهش ریسک**: رصد مداوم تغییرات قانونی و مشاوره حقوقی برای انطباق با الزامات جدید و جلوگیری از جریمه‌های مالی.

### نتیجه‌گیری
استراتژی‌های پیشنهادی باید به‌گونه‌ای طراحی شوند که نتایج مثبت را به حداکثر برسانند و در عین حال ریسک‌ها را به حداقل برسانند. ایجاد یک پورتفولیو متنوع و استفاده از ابزارهای مالی مختلف می‌تواند به کاهش تأثیر نوسانات بر عملکرد کل شرکت کمک کند.